# Project 1, CPSC 585
## Authors: Brian Chu (brianmchu42@csu.fullerton.edu), Jannik Hofmann (jhof@csu.fullerton.edu), Gabriel Villaruel ()

### 1. Clustering (70 points)

#### (a) Data preparation and preprocessing (5 points)
Standardize (z-score scaling) the four chemical properties and save them in a file named
“StdGasProperties.csv”, which will be used for the remaining tasks. Briefly explain:
* The importance of data standardization
* Feature mean and standard deviation before and after standardization

Clustering must be performed using only the four chemical features (excluding idx).

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

df = pd.read_csv("GasProperties.csv")

# skip Idx and only scale the four chemical properties
props = ['T', 'P', 'TC', 'SV']
idx = ['Idx']

preprocessor = ColumnTransformer(
    transformers = [
        ('scaler', StandardScaler(), props),
    ],
    remainder = 'passthrough'
)

transformed = preprocessor.fit_transform(df)

transformed_df = pd.DataFrame(transformed, columns = props + idx)

print('### ORIGINAL ###')
print(df)

print('### MEAN ###')
print(df.mean())
print('### STD ###')
print(df.std())

print('### TRANSFORMED ###')
print(transformed_df)

print('### MEAN ### ')
print(transformed_df.mean())
print('### STD ###')
print(transformed_df.std())

transformed_df.to_csv('StdGasProperties.csv', index=False)

Data standardization is important to regularize the data - numbers that are large in magnitude can be harder to process with some models. For example, since the magnitude of the `T` and `SV` columns in the data are more than two orders of magnitude larger than the `P` and `TC` columns, the `T` and `SV` columns can have an outsize effect on model predictions. Regularizing ensures that all relevant features are considered equally by the model.

As seen in the printout of the data, the original data has some particular mean and standard deviation, but when regularized, the mean becomes 0 and the standard deviation 1 (mathematically, with some floating point rounding error in practice)

#### (b) K-means clustering (10 points)
Perform K-Means with K=3 using and report:
* Initialization method used and initial centroid for each cluster
* Number of iterations until convergence
* Final centroids
* Cluster variances (within-cluster sum of squares)
* Number of samples assigned to each cluster

In [ ]:
from sklearn.cluster import KMeans
from collections import Counter

cluster = KMeans(n_clusters=3, random_state=42)
cluster.fit(transformed_df[props]) # ignore Idx for now

print('Iterations before convergence: ', cluster.n_iter_)
print('Final centroids: ', cluster.cluster_centers_)

# calculate variances
transform = cluster.transform(transformed_df[props]) # gets distances to cluster


print('Samples per cluster: ', Counter(cluster.labels_))


In [ ]:
print(cluster.transform(transformed_df[props]))

#### (c) EM clustering (15 points)
Fit a Gaussian Mixture Model (GMM) with K=3 for clustering and report:
* Initialization method and initial parameters
* Convergence criterion (tolerance or max iterations)
* Covariance type (full, diagonal, spherical) with justification
* Mixture weights (𝜋_𝑘), means (𝜇_𝑘), covariances (Σ_𝑘) for each cluster
* For 3 samples, show probabilities 𝑝(𝑧 = 𝑘|𝑥)

In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np

# initiate
em = GaussianMixture(
    n_components=3,
    init_params='kmeans',
    tol=1e-3,
    max_iter=100,
    covariance_type='full',
    random_state=42
)

# train
em.fit(transformed_df[props])

# print results
print("Converged:", em.converged_)
print("Iterations:", em.n_iter_)
print("Log Likelihood after Convergence:", em.lower_bound_)

print("Weights:", em.weights_)
print("Means:",em.means_)
print("Covariances:", em.covariances_)

sample_probs = em.predict_proba(transformed_df[props].iloc[:3])
print("Sample Probabilities:", np.round(sample_probs, 3))

#### (d) SOM clustering (20 points)

#### (e) Cluster evaluation and interpretation (20 points)

##### Silhouette score

##### Wobbe index

Wobbe indices are used to compare gases for combustion and are derived by dividing the heating value by the square root of the specific gravity. Gases with similar Wobbe indices are considered interchangeable when designing combustion systems. For the purposes of classification, we'll divide the Wobbe indices into three parts (bottom 33% = "regular", middle = "medium", top = "premium")

##### Adjusted rand index

### 2. Classification (50 points)

Split the dataset into training (70%), validation set (15%), and testing set (15%) using a fixed random seed.
All final performance metrics must be reported on the test set.

In [14]:
from sklearn.model_selection import train_test_split

# relabel Idx to Wobbe classification as mentioned above
minIdx, maxIdx = min(transformed_df['Idx']), max(transformed_df['Idx'])
threshold = (maxIdx - minIdx) / 3

def getGrade(idx):
    if idx < minIdx + threshold:
        return 1
    if idx < minIdx + 2 * threshold:
        return 2
    return 3

transformed_df['Idx'] = transformed_df['Idx'].map(getGrade)

# split data into train, val, test
training, vt = train_test_split(transformed_df, test_size=0.3, random_state=42)
validation, testing = train_test_split(vt, test_size=0.5, random_state=42)

#### (a) MLP Classifier (20 points)
Train a MLP to predict quality levels (Regular, Medium, Premium) and report:
* Topology (layers and number of neurons)
* Activation functions
* Optimizer (gradient method)
* Learning rate
* Number of epochs
* Training methods and batch size
* Regularization method if any

Evaluate on test set using confusion matrix, % accuracy, and F1-score.